In [9]:
import sys

sys.path.append("../../")


In [10]:
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session

from models.base import Base
from models.boulder import Boulder
from models.similarity import Similarity



db_path = "matchit.db"

DATABASE_URL = f"sqlite:///../../{db_path}"
engine = create_engine(DATABASE_URL, echo=False)

Base.metadata.create_all(engine)


session = Session(bind=engine)

In [18]:
from sqlalchemy import func

from models.grade import Grade


text = "foundation"

boulder_search = session.execute(
    select(Boulder.id, Boulder.name, Grade.value, func.count(Boulder.ascents))
    .join(Boulder.grade)
    .join(Boulder.ascents)
    .where(Boulder.name_normalized.like(f"%{text}%"))
    .group_by(Boulder.id)
).all()
boulder_search

[(21566, 'Foundations Edge', '8C', 18), (29215, 'Foundation', '7C', 2)]

In [19]:
boulder_id = 21566

boulder = session.get(Boulder, boulder_id)
print("Selected boulder:")
print(boulder)

print("\nSimilar boulders:")
similarities = session.execute(
    select(Boulder, Similarity.score)
    .join(Similarity, Boulder.id == Similarity.id2)
    .where(Similarity.id1 == boulder_id)
    .order_by(Similarity.score.desc())
    .limit(20)
).all()
for boulder, score in similarities:
    print(boulder, score)

Selected boulder:
<Boulder(name: Foundations Edge, grade: 8C, Ascents: 18)>

Similar boulders:
<Boulder(name: Scarred for Life, grade: 8B+, Ascents: 24)> 2
<Boulder(name: Compass North, grade: 8B+, Ascents: 14)> 1.6666667461395264
<Boulder(name: Big nose , grade: 8C, Ascents: 6)> 1.6666667461395264
<Boulder(name: Harder Better Faster, grade: 8B+, Ascents: 4)> 1.1111111640930176
<Boulder(name: Radja, grade: 8B+, Ascents: 26)> 1.1111111640930176
<Boulder(name: Zima Blue, grade: 8B, Ascents: 15)> 1.1111111640930176
<Boulder(name: The Bitter End, grade: 8B+, Ascents: 7)> 0.9523809552192688
<Boulder(name: Fuck the system, grade: 8C+, Ascents: 3)> 0.8333333730697632
<Boulder(name: Le son du poète, grade: 8B, Ascents: 17)> 0.6451612710952759
<Boulder(name: Television addict, grade: 8B, Ascents: 9)> 0.625
<Boulder(name: Solitary daze, grade: 8C, Ascents: 2)> 0.5555555820465088
<Boulder(name: Permanent midnight low, grade: 8C+, Ascents: 2)> 0.5555555820465088
<Boulder(name: Touched by the devil

In [16]:
boulder_id = 21391

boulder = session.get(Boulder, boulder_id)

print(f"{'='*60}")
print(f"Boulder: {boulder.name}")
print(f"{'='*60}")
print(f"ID:              {boulder.id}")
print(f"Grade:           {boulder.grade.value if boulder.grade else 'N/A'}")
print(f"Normalized Name: {boulder.name_normalized}")
print(f"Total Ascents:   {len(boulder.ascents)}")
print(f"Crag:            {boulder.crag.name if boulder.crag else 'N/A'}")
print(f"Area:            {boulder.crag.area.name if boulder.crag and boulder.crag.area else 'N/A'}")
print(f"Similarity idx:  {boulder.similarity_matrix_id if boulder.similarity_matrix_id is not None else 'N/A'}")
print(f"\n{'Ascents Details:':-^60}")
for i, ascent in enumerate(boulder.ascents[:10], 1):
    print(f"  {i}. User ID: {ascent.user_id}, Date: {ascent.log_date if hasattr(ascent, 'log_date') else 'N/A'}")
if len(boulder.ascents) > 10:
    print(f"  ... and {len(boulder.ascents) - 10} more ascents")
print(f"{'='*60}")

Boulder: Les yeux rouges
ID:              21391
Grade:           7C
Normalized Name: les yeux rouges
Total Ascents:   94
Crag:            Vernayaz
Area:            Bas-Valais
Similarity idx:  729

----------------------Ascents Details:----------------------
  1. User ID: 1060, Date: 2025-12-27
  2. User ID: 476, Date: 2025-12-25
  3. User ID: 86, Date: 2025-12-23
  4. User ID: 1669, Date: 2025-12-21
  5. User ID: 10390, Date: 2025-12-13
  6. User ID: 1443, Date: 2025-12-12
  7. User ID: 1055, Date: 2025-11-22
  8. User ID: 585, Date: 2025-03-19
  9. User ID: 1648, Date: 2025-03-08
  10. User ID: 3105, Date: 2025-03-06
  ... and 84 more ascents


In [14]:
from ml.crud import delete_similarity_data


delete_similarity_data(session=session, area_slug="ticino")